In [1]:
from retrievers.vector_retriever import VectorRetriever
from retrievers.fulltext_retriever import FullTextRetriever
from retrievers.hybrid_retriever import HybridRetriever
from retrievers.text2cypher import Text2CypherRetriever
from graph.neo4j_manager import Neo4jManager
from agents.multi_agent_system import MultiAgentSystem
from agents.question_descomposer import QuestionDecomposer
import pandas as pd


In [2]:
neo4j = Neo4jManager()


In [3]:
dec = QuestionDecomposer()


In [20]:
query = """
MATCH (a:Character)-[:HAS_ROLE]->(r:Role) WHERE toLower(r.name) CONTAINS "helper" WITH collect('Character: ' + a.name + '\nDescription: ' + a.description) AS characters RETURN reduce(result = '', c IN characters | result + CASE WHEN result = '' THEN '' ELSE '\n\nOR\n\n' END + c) AS ground_truth
"""

results = neo4j.execute_query(query)

print(results)



[{'ground_truth': "Character: monkey\nDescription: The monkey is a companion of Momotaro. He goes to the Ogres' Island with him.\n\nOR\n\nCharacter: pheasant\nDescription: The pheasant is a companion of Momotaro. He flies over the castle gate to peck the Ogres.\n\nOR\n\nCharacter: dog\nDescription: The dog is a companion of Momotaro. He breaks the bolts and bars with him.\n\nOR\n\nCharacter: crow\nDescription: A clever and resourceful bird who helps the bird catcher in his tasks and becomes a trusted advisor to the emperor."}]


In [5]:
dataset = pd.read_csv('./data/benchmark_data.csv', delimiter=";")
print(f"Benchmark CSV: {len(dataset)} preguntas")
dataset.head()


Benchmark CSV: 36 preguntas


,question,cypher,type
0,Hello,"RETURN ""Hello! I'm a knowledge assistant focus...",greeting
1,What can you do?,"RETURN ""I can answer questions about folktales...",greeting
2,How is the weather today?,"RETURN ""This question is outside my scope."" AS...",out_of_scope
3,Who won the last football World Cup?,"RETURN ""This question is outside my scope."" AS...",out_of_scope
4,What is the capital of France?,RETURN 'This question is outside my scope.' AS...,out_of_scope


In [6]:
for i, row in dataset.iterrows():
	question = row["question"]

	subquestions = dec.run(question)	
	print(f"{question}: {subquestions}")


Hello: ['Hello']
What can you do?: ['What can you do?']
How is the weather today?: ['How is the weather today?']
Who won the last football World Cup?: ['Who won the last football World Cup?']
What is the capital of France?: ['What is the capital of France?']
Which movies did Momotaro appear in?: ['Which movies did Momotaro appear in?']
What is Momotaro's surname?: ["What is Momotaro's surname?"]
Hold old is Momotaro?: ['Hold old is Momotaro?']
What is the town population in 'Sharing Joy and Sorrow'?: ["What is the town population in 'Sharing Joy and Sorrow'?"]
List all Genre nodes in the graph: ['List all Genre nodes in the graph']
How many Folktale nodes are in the graph?: ['How many Folktale nodes are in the graph?']
How many Event nodes are in the graph?: ['How many Event nodes are in the graph?']
How many Character nodes are in the graph?: ['How many Character nodes are in the graph?']
How many Place nodes are in the graph?: ['How many Place nodes are in the graph?']
How many Objec

In [7]:
for i, row in dataset.iterrows():
	question = row["question"]
	cypher = row["cypher"]
	
	result = neo4j.execute_query(cypher)
	print(f"{question}: {result}")
	

Hello: [{'ground_truth': "Hello! I'm a knowledge assistant focused on folktales from all around the world. You can ask me about their characters, relationships, themes or how they are structured."}]
What can you do?: [{'ground_truth': 'I can answer questions about folktales: their narrative structure, the characters that make them up and their features, the relevant places and objects that appear, among others.'}]
How is the weather today?: [{'ground_truth': 'This question is outside my scope.'}]
Who won the last football World Cup?: [{'ground_truth': 'This question is outside my scope.'}]
What is the capital of France?: [{'ground_truth': 'This question is outside my scope.'}]
Which movies did Momotaro appear in?: [{'ground_truth': 'This question is outside my scope.'}]
What is Momotaro's surname?: [{'ground_truth': 'This information is not in the knowledge base.'}]
Hold old is Momotaro?: [{'ground_truth': 'This information is not in the knowledge base.'}]
What is the town population i

In [8]:
query = "Who is Momotaro?"

extra_match = """
OPTIONAL MATCH (node)-[:HAS_ROLE]->(r:Role)
"""

return_projection={
	"name": "node.name",
	"description": "node.description",
	"role": "r.name"
}
retriever = VectorRetriever(neo4j, "character_embeddings", return_projection, extra_match)
results = retriever.retrieve(query)

print(f"Vector search - {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r["score"]:.3f}")
    print(f"\t{r["name"]} [{r["role"]}]: {r["description"]}\n")


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL db.index.vector.queryNodes($index_name, $top_k, $query_embedding)\n        YIELD node, score\n\n        \nOPTIONAL MATCH (node)-[:HAS_ROLE]->(r:Role)\n\n\n        RETURN node.name AS name, node.description AS description, r.name AS role, score\n        ORDER BY score DESC\n        '


Vector search - 2 resultados

1. score=0.866
	Momotaro [main_character]: Momotaro is the son of the old couple. He goes on a journey to get treasure from the Ogres' Island.

2. score=0.831
	dog [helper]: The dog is a companion of Momotaro. He breaks the bolts and bars with him.



In [9]:
query = "millet dumpling"
return_fields = {
    "description": "node.description",
    "name": "node.name"
}
retriever = FullTextRetriever(neo4j, "event_fulltext", "Event", "description", return_fields)
results = retriever.retrieve(query)

print(f"Fulltext search - {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r["score"]:.3f}")
    print(f"\t{r["name"]}: {r["description"]}\n")


Fulltext search - 2 resultados

1. score=3.452
	Momotaro gives dog a dumpling.: “I’ve got some of the best millet dumplings in all Japan.”“Give me one,” says the dog, “and I will go with you.”So Momotaro gave a millet dumpling to the dog, and the four of them jogged on together.

2. score=3.452
	Momotaro Acquires Pheasant Ally: “I’ve got some of the best millet dumplings in all Japan.”“Give me one,” says the pheasant, “and I will go with you.”So Momotaro gave a millet dumpling to the pheasant, and the three of them jogged on together.



In [10]:
query = "How did Momotaro convince the animals to join him on his journey?"
return_fields = {
    "description": "node.description",
    "name": "node.name"
}
retriever = HybridRetriever(neo4j, "character_embeddings", "character_fulltext", "Character", "description", vector_weight=0.5, return_projection=return_fields, include_score=True)
results = retriever.retrieve(query, 10)

print(f"Hybrid search - {len(results)} resultados\n")
for i, r in enumerate(results, 1):
    print(f"{i}. score={r["score"]:.3f}")
    print(f"\t{r["name"]}: {r["description"]}\n")


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=4, column=13, offset=60>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 60, 'line': 4, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL () {\n            // Vector search\n            CALL db.index.vector.queryNodes($vector_index, $top_k, $query_embedding)\n            YIELD node, score\n            \n            WITH collect({node: node, score: score}) AS nodes, max(score) AS maxScore, min(score) AS minScore\n            UNWIND nodes AS n\n            RET

Hybrid search - 10 resultados

1. score=0.922
	monkey: The monkey is a companion of Momotaro. He goes to the Ogres' Island with him.

2. score=0.894
	dog: The dog is a companion of Momotaro. He breaks the bolts and bars with him.

3. score=0.823
	Momotaro: Momotaro is the son of the old couple. He goes on a journey to get treasure from the Ogres' Island.

4. score=0.495
	pheasant: The pheasant is a companion of Momotaro. He flies over the castle gate to peck the Ogres.

5. score=0.433
	ogre: The ogres are the enemies of Momotaro and his companions. They are overthrown by them.

6. score=0.205
	old man: The old man is a kind and poor man who lives with his wife. He goes to the mountains to gather firewood.

7. score=0.140
	crow: A clever and resourceful bird who helps the bird catcher in his tasks and becomes a trusted advisor to the emperor.

8. score=0.101
	the tailor: A quarrelsome and abusive husband who beats his wife.

9. score=0.072
	cock: A minor character who is heard crowing i

In [ ]:
text2cypher = Text2CypherRetriever(neo4j)


In [11]:
rag = MultiAgentSystem(neo4j, verbose=True)


In [ ]:
result = rag.answer("Give me a character who has the role of helper and then describe them.", "test30", max_iterations=1, decompose=False)

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)



========== DECOMPOSED QUERIES ==========

  ========== Query 1 ==========
  Give me a character who has the role of helper and then describe them.

========== ITERATION 1 ==========

  ========== QUESTION 1 ==========
  Question: Give me a character who has the role of helper and then describe them.



  ========== AI -> TOOL CALL ==========
  [
  {
    "name": "character_tool",
    "args": {
      "query": "Give me a character who has the role of helper and then describe them"
    },
    "id": "22c8a38b-02de-457a-8a18-0ba958578ad1",
    "type": "tool_call"
  }
]

========== DECOMPOSED QUERIES ==========

  ========== Query 1 ==========
  Get all characters with the role 'helper'

  ========== Query 2 ==========
  Describe each character retrieved in step 1

========== ITERATION 1 ==========

  ========== QUESTION 1 ==========
  Question: Get all characters with the role 'helper'



  ========== AI -> TOOL CALL ==========
  [
  {
    "name": "text2cypher",
    "args": {
      "query":

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=9, offset=9>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 9, 'line': 2, 'column': 9}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL db.index.vector.queryNodes($index_name, $top_k, $query_embedding)\n        YIELD node, score\n\n        \nOPTIONAL MATCH (node)-[:HAS_ROLE]->(r:Role)\n\n\n        RETURN node.name AS name, node.description AS description, r.name AS role\n        ORDER BY score DESC\n        '





    ========== TOOL -> RESPONSE (vector_search) ==========
    [
  {
    "type": "text",
    "text": "[\"name: pheasant\\ndescription: The pheasant is a companion of Momotaro. He flies over the castle gate to peck the Ogres.\\nrole: helper\"]"
  }
]



  ========== AI -> FINAL ANSWER ==========
  The characters with the role 'helper' are:

1. Monkey
   - Description: The monkey is a companion of Momotaro. He goes to the Ogres' Island with him.
2. Pheasant
   - Description: The pheasant is a companion of Momotaro. He flies over the castle gate to peck the Ogres.
3. Dog
   - Description: The dog is a companion of Momotaro. He breaks the bolts and bars with him.

  ========== CONTEXT ==========
  [
  "name: pheasant\ndescription: The pheasant is a companion of Momotaro. He flies over the castle gate to peck the Ogres.\nrole: helper"
]

  ========== CRITIQUE ==========
  {
  "is_complete": true,
  "is_faithful": true,
  "missing_info": []
}

========== LOOP FINISHED ==========



    ==

In [27]:
result = rag.answer("Which characters appear in the folktale 'The Thieves and the Cock'?", "test25", max_iterations=1, decompose=False)

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)




========== DECOMPOSED QUERIES ==========

  ========== Query 1 ==========
  Which characters appear in the folktale 'The Thieves and the Cock'?

========== ITERATION 1 ==========

  ========== QUESTION 1 ==========
  Question: Which characters appear in the folktale 'The Thieves and the Cock'?



  ========== AI -> TOOL CALL ==========
  [
  {
    "name": "character_tool",
    "args": {
      "query": "Which characters appear in the folktale ‘The Thieves and the Cock’"
    },
    "id": "1db7925b-a05e-4d62-8c4e-d88f304622ce",
    "type": "tool_call"
  }
]

========== DECOMPOSED QUERIES ==========

  ========== Query 1 ==========
  Which characters appear in the folktale ‘The Thieves and the Cock’

========== ITERATION 1 ==========

  ========== QUESTION 1 ==========
  Question: Which characters appear in the folktale ‘The Thieves and the Cock’



  ========== AI -> TOOL CALL ==========
  [
  {
    "name": "hybrid_search",
    "args": {
      "query": "characters in The Thieves and the 

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=4, column=13, offset=60>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 60, 'line': 4, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL () {\n            // Vector search\n            CALL db.index.vector.queryNodes($vector_index, $top_k, $query_embedding)\n            YIELD node, score\n            \n            WITH collect({node: node, score: score}) AS nodes, max(score) AS maxScore, min(score) AS minScore\n            UNWIND nodes AS n\n            RET




    ========== TOOL -> RESPONSE (hybrid_search) ==========
    [
  {
    "type": "text",
    "text": "[\"name: Thief\\ndescription: A person who broke into a house and stole the Cock.\\nrole: primary_character\"]"
  }
]



  ========== AI -> FINAL ANSWER ==========
  The characters that appear in the folktale 'The Thieves and the Cock' are:

1. The Thief
2. The Cock

  ========== CONTEXT ==========
  [
  "name: Thief\ndescription: A person who broke into a house and stole the Cock.\nrole: primary_character"
]

  ========== CRITIQUE ==========
  {
  "is_complete": true,
  "is_faithful": true,
  "missing_info": []
}

========== LOOP FINISHED ==========



    ========== TOOL -> RESPONSE (character_tool) ==========
    [
  {
    "type": "text",
    "text": "{\"answer\": \"The characters that appear in the folktale 'The Thieves and the Cock' are:\\n\\n1. The Thief\\n2. The Cock\", \"context\": [\"name: Thief\\ndescription: A person who broke into a house and stole the Cock.\\nrole: prim

In [14]:
result = rag.answer("How many Character nodes are in the graph?", "duh")

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)



========== DECOMPOSED QUERIES ==========

  ========== Query 1 ==========
  How many Character nodes are in the graph?

========== ITERATION 1 ==========

  ========== QUESTION 1 ==========
  Question: How many Character nodes are in the graph?



  ========== AI -> TOOL CALL ==========
  [
  {
    "name": "folktale_tool",
    "args": {
      "query": "How many Character nodes are in the graph?"
    },
    "id": "cef9794a-b3d5-441f-a098-f986624a33ea",
    "type": "tool_call"
  }
]



    ========== TOOL -> RESPONSE (folktale_tool) ==========
    [
  {
    "type": "text",
    "text": "{\"cypher\": \"MATCH (n:Character) RETURN count(n)\", \"context\": [\"{'count(n)': 24}\"]}"
  }
]



  ========== AI -> FINAL ANSWER ==========
  There are 24 Character nodes in the graph.

  ========== CONTEXT ==========
  [
  "{'count(n)': 24}"
]

  ========== CRITIQUE ==========
  {
  "is_complete": true,
  "is_faithful": true,
  "missing_info": []
}

========== LOOP FINISHED ==========

Question: How 

In [17]:
result = rag.answer("Who is matt?", "matt", max_iterations=1, decompose=False)

print("\nQuestion:", result.question)
if result.iterations:
	print("Context:", result.iterations[-1].context)
print("Answer:", result.answer)



========== DECOMPOSED QUERIES ==========

  ========== Query 1 ==========
  Who is matt?

========== ITERATION 1 ==========

  ========== QUESTION 1 ==========
  Question: Who is matt?



  ========== AI -> TOOL CALL ==========
  [
  {
    "name": "character_tool",
    "args": {
      "query": "Who is matt?"
    },
    "id": "20453c5b-86ef-4f28-99fd-21eb2073cb0d",
    "type": "tool_call"
  }
]

========== DECOMPOSED QUERIES ==========

  ========== Query 1 ==========
  Who is matt?

========== ITERATION 1 ==========

  ========== QUESTION 1 ==========
  Question: Who is matt?



  ========== AI -> TOOL CALL ==========
  [
  {
    "name": "hybrid_search",
    "args": {
      "query": "Who is Matt?"
    },
    "id": "ed441246-8c9d-441d-94ef-f5820045b28a",
    "type": "tool_call"
  }
]


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=4, column=13, offset=60>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 60, 'line': 4, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        CALL () {\n            // Vector search\n            CALL db.index.vector.queryNodes($vector_index, $top_k, $query_embedding)\n            YIELD node, score\n            \n            WITH collect({node: node, score: score}) AS nodes, max(score) AS maxScore, min(score) AS minScore\n            UNWIND nodes AS n\n            RET




    ========== TOOL -> RESPONSE (hybrid_search) ==========
    [
  {
    "type": "text",
    "text": "[\"name: goat\\ndescription: A foolish and scatterbrained character who fails to impress the tiger.\\nrole: secondary_character\"]"
  }
]



  ========== AI -> FINAL ANSWER ==========
  Matt is a secondary character in the story, known as the goat. He is described as foolish and scatterbrained, failing to impress the tiger.

  ========== CONTEXT ==========
  [
  "name: goat\ndescription: A foolish and scatterbrained character who fails to impress the tiger.\nrole: secondary_character"
]

  ========== CRITIQUE ==========
  {
  "is_complete": true,
  "is_faithful": true,
  "missing_info": []
}

========== LOOP FINISHED ==========



    ========== TOOL -> RESPONSE (character_tool) ==========
    [
  {
    "type": "text",
    "text": "{\"answer\": \"Matt is a secondary character in the story, known as the goat. He is described as foolish and scatterbrained, failing to impress the tiger

In [ ]:
neo4j.close()
